# NQ Opening Range Breakout (ORB) – Backtest and Optimization

Rules as described in "Best NQ Strategy 2026 – Master the Opening Range Breakout" (Trading Mania):

* **Opening range:** high / low of the first 15 minutes after the New York open (9:30–9:45 ET).
* **Entry:** a **5-minute candle closes** above the OR high (long) / below the OR low (short), volume of that candle above
  average; entry on the **next candle open** or on a **retest** of the broken level.
* **Stop:** 5–10 points beyond the broken level (long: below the OR high).
* **Target:** 2:1 or 3:1 (preferred), alternatively previous-day high/low, or 50 % at 2R with a runner to 4R.
* **False breakout:** if price closes back inside the range within 2 candles → exit.
* **Timing:** 9:45–11:30 primary window, 13:00–15:30 second window, no new entries after 15:30; the article does not say
  when to close – here every position is closed at **15:55 ET** (flat at the end of the day, as prop firms require).
* **After a win: stop for the day** (one trade per day by default).

**Not testable with these data:** the news filter (NFP, CPI, Fed – no calendar in the data) and discretionary judgement
of "strong momentum".

**Honest optimization:** the data are split by time into **optimization** (all but the last 24 months), **validation**
(the 12 months before the test) and **test** (last 12 months). Every parameter combination is simulated; the choice uses
only optimization + validation (`min(t-value optimization, t-value validation)`), and is then applied **once** to the test
period, compared with the **article's default rules** and with **random direction** on the same entries.

## 0. Configuration

In [ ]:
from pathlib import Path

SYMBOL = "NQ"
DATA_PATH = Path(r"E:\Programierung\Daten\NQ\Candelexport_NQ_M1_MERGED_12.12.2019_26.10.2026")  # 1-minute data
TEST_MONTHS = 12                 # last 12 months = test period (used once at the end)
VALIDATION_MONTHS = 12           # the 12 months before = validation period
OUTPUT_DIR = Path("orb_results_NQ")

# Contract and costs
TICK_SIZE = 0.25                 # NQ points per tick
POINT_VALUE_USD = 20.0           # NQ: 20 USD per point (MNQ: 2 USD)
COST_PER_TRADE_POINTS = 0.5      # round trip incl. spread/slippage and commission (= 2 ticks) -> adjust to your account
TARGET_FILL_EXTRA_TICKS = 1      # a target counts as filled only if the price trades 1 tick beyond it

# Session (New York time, daylight saving time handled automatically)
RTH_START = (9, 30)
FLAT_TIME = (15, 55)             # every open position is closed here
WINDOWS = {                      # entry windows by the close time of the breakout candle
    "primary": [((9, 45), (11, 30))],
    "primary+afternoon": [((9, 45), (11, 30)), ((13, 0), (15, 30))],
}
MAX_TRADES_PER_DAY = 1           # article: after a win, stop for the day
STOP_AFTER_WIN = True
VOLUME_LOOKBACK_DAYS = 20        # "above average volume" = above the average of the same 5-minute slot over 20 days
FALSE_BREAKOUT_BARS = 2          # exit if a 5-minute close is back inside the range within this many candles
RETEST_MINUTES = 30              # retest entry: limit at the broken level, valid this long after the breakout candle
RETEST_FILL_THROUGH_TICKS = 1    # the limit counts as filled only if the price trades this many ticks THROUGH the level
COST_SENSITIVITY_POINTS = [0.5, 1.0, 1.5]   # test result of the chosen combination with higher costs
MIN_RISK_POINTS = 1.0            # skip trades with a smaller stop distance

# The article's rules (baseline, no optimization)
ARTICLE_RULES = dict(or_minutes=15, entry="next_open", volume_filter=True, stop="level_10", target="3R",
                     false_breakout_exit=True, window="primary+afternoon", side="both")

# Parameter grid for the optimization
GRID = dict(
    or_minutes=[5, 15, 30],
    entry=["next_open", "retest"],
    volume_filter=[True, False],
    stop=["level_5", "level_10", "opposite"],           # 5 / 10 points beyond the level, or the other side of the range
    target=["1.5R", "2R", "3R", "partial_2R_4R", "prev_day"],
    false_breakout_exit=[True, False],
    window=["primary", "primary+afternoon"],
    side=["both", "long", "short"],
)
MIN_TRADES_OPT = 100             # combinations with fewer trades are not eligible
MIN_TRADES_VAL = 30
RANDOM_RUNS = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
n_combos = 1
for v in GRID.values():
    n_combos *= len(v)
print(f"Grid: {n_combos:,} combinations")

## 1. Imports

In [ ]:
import itertools
import time

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

## 2. Load 1-Minute Data

In [ ]:
def resolve_data_files(path: Path) -> list[Path]:
    if path.is_file():
        return [path]
    with_ext = Path(str(path) + ".csv")
    if with_ext.is_file():
        return [with_ext]
    if path.is_dir():
        files = sorted(path.glob("*.csv"))
        if files:
            return files
    raise FileNotFoundError(f"No CSV found at {path} (also tried {with_ext})")


parts = []
for f in resolve_data_files(DATA_PATH):
    raw = pd.read_csv(f, sep=";", encoding="utf-8-sig", usecols=["Time UTC", "Open", "High", "Low", "Close", "Volume"])
    parts.append(raw)
    print(f"{f.name}: {len(raw):,} rows")
raw = pd.concat(parts, ignore_index=True)
df = pd.DataFrame({
    "timestamp": pd.to_datetime(raw["Time UTC"], format="%d.%m.%Y %H:%M:%S", utc=True, errors="coerce"),
    **{c.lower(): pd.to_numeric(raw[c], errors="coerce") for c in ["Open", "High", "Low", "Close", "Volume"]},
})
df = df.dropna(subset=["timestamp", "open", "high", "low", "close"]).sort_values("timestamp", kind="stable")
df = df.drop_duplicates("timestamp", keep="last").reset_index(drop=True)
df["volume"] = df["volume"].fillna(0.0)
print(f"{len(df):,} one-minute candles, {df['timestamp'].min()} → {df['timestamp'].max()}")

## 3. Trading Days (Regular Session, New York Time)

Per day: 1-minute candles from 9:30 to 16:00 ET, 5-minute candles built from them (aligned to 9:30), previous-day high/low
and the average volume of every 5-minute slot over the previous `VOLUME_LOOKBACK_DAYS` days.

In [ ]:
RTH_START_MIN = RTH_START[0] * 60 + RTH_START[1]
FLAT_MIN = FLAT_TIME[0] * 60 + FLAT_TIME[1]
N_SLOTS = (16 * 60 - RTH_START_MIN) // 5


def to_min(hm):
    return hm[0] * 60 + hm[1]


et = df["timestamp"].dt.tz_convert("America/New_York")
mod_all = (et.dt.hour * 60 + et.dt.minute).to_numpy()
rth = (mod_all >= RTH_START_MIN) & (mod_all < 16 * 60)
sub = df[rth].reset_index(drop=True)
sub_mod = mod_all[rth]
sub_date = et[rth].dt.date.to_numpy()
starts = np.flatnonzero(np.r_[True, sub_date[1:] != sub_date[:-1]])
ends = np.r_[starts[1:], len(sub)]

cols = {k: sub[k].to_numpy("float64") for k in ["open", "high", "low", "close", "volume"]}
ts_all = sub["timestamp"].to_numpy()
days = []
for s0, e0 in zip(starts, ends):
    mod = sub_mod[s0:e0].copy()
    if mod[0] > RTH_START_MIN + 5 or len(mod) < 60:          # incomplete day (holiday, data gap)
        continue
    o, h, l, c, v = (cols[k][s0:e0].copy() for k in ["open", "high", "low", "close", "volume"])
    slot = (mod - RTH_START_MIN) // 5
    b_start = np.flatnonzero(np.r_[True, slot[1:] != slot[:-1]])
    b_end = np.r_[b_start[1:], len(slot)] - 1                 # index of the last 1-minute candle in each 5-minute candle
    days.append({
        "date": sub_date[s0], "ts": ts_all[s0:e0].copy(), "mod": mod,
        "o": o, "h": h, "l": l, "c": c,
        "b_slot": slot[b_start], "b_end": b_end, "b_endmod": RTH_START_MIN + 5 * (slot[b_start] + 1),
        "b_close": c[b_end], "b_vol": np.add.reduceat(v, b_start),
        "rth_high": h.max(), "rth_low": l.min(),
        "flat_idx": int(np.searchsorted(mod, FLAT_MIN) - 1),
    })
del cols, ts_all

# previous day high / low and average volume per 5-minute slot (previous days only)
vol_matrix = np.full((len(days), N_SLOTS), np.nan)
for i, d in enumerate(days):
    d["prev_high"] = days[i - 1]["rth_high"] if i else np.nan
    d["prev_low"] = days[i - 1]["rth_low"] if i else np.nan
    vol_matrix[i, d["b_slot"]] = d["b_vol"]
slot_avg = pd.DataFrame(vol_matrix).rolling(VOLUME_LOOKBACK_DAYS, min_periods=5).mean().shift(1).to_numpy()
for i, d in enumerate(days):
    avg = slot_avg[i, d["b_slot"]]
    d["vol_ok"] = np.where(np.isnan(avg), True, d["b_vol"] > avg)

# opening range per length
for d in days:
    d["or"] = {}
    for m in GRID["or_minutes"]:
        mask = d["mod"] < RTH_START_MIN + m
        d["or"][m] = (d["h"][mask].max(), d["l"][mask].min()) if mask.sum() >= 0.8 * m else None

# periods
day_dates = pd.to_datetime(pd.Series([d["date"] for d in days]))
test_start = day_dates.max() - pd.DateOffset(months=TEST_MONTHS)
val_start = test_start - pd.DateOffset(months=VALIDATION_MONTHS)
period_of_day = np.where(day_dates >= test_start, "test", np.where(day_dates >= val_start, "validation", "optimization"))
for d, p in zip(days, period_of_day):
    d["period"] = p
for p in ["optimization", "validation", "test"]:
    sel = day_dates[period_of_day == p]
    print(f"{p:<12}: {len(sel):>4} trading days, {sel.min().date()} → {sel.max().date()}")

## 4. Trade Simulation

Exits are resolved on 1-minute candles: stop and target in the same candle → stop (conservative); target only with
`TARGET_FILL_EXTRA_TICKS` beyond it; false-breakout exit at the close of the 5-minute candle; otherwise flat at 15:55.
**Retest (limit) entries:** filled only if the price trades `RETEST_FILL_THROUGH_TICKS` through the level, and the target
can only be reached from the **next** minute on – within the fill minute it is unknown whether the high came before or
after the fill.
With `partial_2R_4R`: half at 2R, stop of the runner to break-even, runner target 4R.

In [ ]:
INF = 10 ** 9
EXTRA = TARGET_FILL_EXTRA_TICKS * TICK_SIZE


def first_true(x):
    return int(np.argmax(x)) if x.any() else INF


def run_exit(d, direction, s, entry, stop, target, fb_idx=INF, target_from=0):
    """Exit of one position from minute index s. Returns (exit_idx, exit_price, reason).
    target_from=1: the target cannot be reached in the entry minute (limit fills: the minute's high/low may have
    happened BEFORE the fill, so counting it would look into the past)."""
    e = d["flat_idx"]
    if s > e:
        return None
    hh, ll = d["h"][s:e + 1], d["l"][s:e + 1]
    if direction == 1:
        k_stop, k_tgt = first_true(ll <= stop), first_true(hh[target_from:] >= target + EXTRA)
    else:
        k_stop, k_tgt = first_true(hh >= stop), first_true(ll[target_from:] <= target - EXTRA)
    if k_tgt < INF:
        k_tgt += target_from
    k_fb = fb_idx - s if fb_idx < INF and fb_idx >= s else INF
    k_flat = e - s
    if k_stop <= min(k_tgt, k_fb, k_flat):
        return s + k_stop, stop, "stop"
    if k_tgt <= min(k_fb, k_flat):
        return s + k_tgt, target, "target"
    if k_fb <= k_flat:
        return s + k_fb, d["c"][s + k_fb], "false_breakout"
    return e, d["c"][e], "flat_eod"


def trade_result(d, direction, s, entry, risk, target_rule, level_target, fb_idx, target_from=0):
    """Simulate one trade incl. target rule; returns (exit_idx, points, reason)."""
    stop = entry - direction * risk
    if target_rule == "partial_2R_4R":
        r1 = run_exit(d, direction, s, entry, stop, entry + direction * 2 * risk, fb_idx, target_from)
        if r1 is None:
            return None
        if r1[2] != "target":
            return r1[0], direction * (r1[1] - entry), r1[2]
        k = r1[0]
        r2 = run_exit(d, direction, k + 1, entry, entry, entry + direction * 4 * risk) if k + 1 <= d["flat_idx"] else None
        runner = direction * ((r2[1] if r2 else d["c"][k]) - entry)
        return (r2[0] if r2 else k), 0.5 * 2 * risk + 0.5 * runner, "partial+" + (r2[2] if r2 else "flat_eod")
    if target_rule == "prev_day":
        tgt = level_target if (level_target is not None and not np.isnan(level_target)
                               and direction * (level_target - entry) >= risk) else entry + direction * 2 * risk
    else:
        tgt = entry + direction * float(target_rule[:-1]) * risk
    r = run_exit(d, direction, s, entry, stop, tgt, fb_idx, target_from)
    return None if r is None else (r[0], direction * (r[1] - entry), r[2])


def candidate_bars(d, cfg):
    """5-minute breakout candles of a day for one configuration: list of (bar_pos, direction)."""
    orr = d["or"][cfg["or_minutes"]]
    if orr is None:
        return [], None
    orh, orl = orr
    after_or = (d["b_endmod"] - 5) >= RTH_START_MIN + cfg["or_minutes"]
    in_window = np.zeros(len(d["b_endmod"]), bool)
    for a, b in WINDOWS[cfg["window"]]:
        in_window |= (d["b_endmod"] >= to_min(a)) & (d["b_endmod"] <= to_min(b))
    ok = after_or & in_window & (d["vol_ok"] if cfg["volume_filter"] else True)
    longs = ok & (d["b_close"] > orh) & (cfg["side"] in ("both", "long"))
    shorts = ok & (d["b_close"] < orl) & (cfg["side"] in ("both", "short"))
    pos = np.flatnonzero(longs | shorts)
    return [(int(p), 1 if longs[p] else -1) for p in pos], (orh, orl)


def backtest(cfg, day_list, direction_mode="rules", seed=0):
    """direction_mode: 'rules' (breakout direction) or 'random' (same entries, coin-flip direction, no false-breakout exit)."""
    rng = np.random.default_rng(seed)
    stop_kind, buf = (cfg["stop"].split("_") + ["0"])[:2]
    rows = []
    for d in day_list:
        cands, orr = candidate_bars(d, cfg)
        if not cands:
            continue
        orh, orl = orr
        n_trades, busy_until = 0, -1
        for p, sig_dir in cands:
            if n_trades >= MAX_TRADES_PER_DAY:
                break
            end_idx = d["b_end"][p]
            if end_idx <= busy_until:
                continue
            level = orh if sig_dir == 1 else orl
            # entry
            if cfg["entry"] == "next_open":
                s = end_idx + 1
                if s > d["flat_idx"]:
                    continue
                entry = d["o"][s]
            else:
                last = int(np.searchsorted(d["mod"], d["b_endmod"][p] + RETEST_MINUTES, side="right") - 1)
                last = min(last, d["flat_idx"])
                rng_idx = np.arange(end_idx + 1, last + 1)
                if not len(rng_idx):
                    continue
                through = RETEST_FILL_THROUGH_TICKS * TICK_SIZE
                touch = d["l"][rng_idx] <= level - through if sig_dir == 1 else d["h"][rng_idx] >= level + through
                if not touch.any():
                    continue
                s = int(rng_idx[np.argmax(touch)])
                entry = min(d["o"][s], level) if sig_dir == 1 else max(d["o"][s], level)
            # stop distance from the rules (always relative to the breakout direction)
            if stop_kind == "level":
                stop = level - sig_dir * float(buf)
            else:
                stop = orl if sig_dir == 1 else orh
            risk = sig_dir * (entry - stop)
            if risk < MIN_RISK_POINTS:
                continue
            direction = sig_dir if direction_mode == "rules" else int(rng.choice([-1, 1]))
            # false breakout: 5-minute close back inside the range within FALSE_BREAKOUT_BARS candles
            fb_idx = INF
            if cfg["false_breakout_exit"] and direction_mode == "rules":
                first_bar = int(np.searchsorted(d["b_end"], s))
                for q in range(first_bar, min(first_bar + FALSE_BREAKOUT_BARS, len(d["b_end"]))):
                    back_inside = d["b_close"][q] < orh if sig_dir == 1 else d["b_close"][q] > orl
                    if back_inside:
                        fb_idx = int(d["b_end"][q])
                        break
            level_target = (d["prev_high"] if direction == 1 else d["prev_low"])
            res = trade_result(d, direction, s, entry, risk, cfg["target"], level_target, fb_idx,
                               target_from=1 if cfg["entry"] == "retest" else 0)
            if res is None:
                continue
            exit_idx, points, reason = res
            net = points - COST_PER_TRADE_POINTS
            rows.append({
                "date": d["date"], "period": d["period"], "entry_time": d["ts"][s], "exit_time": d["ts"][exit_idx],
                "direction": direction, "entry": entry, "risk_points": risk, "points": points,
                "net_points": net, "net_R": net / risk, "net_usd": net * POINT_VALUE_USD, "exit_reason": reason,
            })
            n_trades += 1
            busy_until = exit_idx
            if STOP_AFTER_WIN and net > 0:
                break
    return pd.DataFrame(rows)


def metrics(tr: pd.DataFrame, n_days: int) -> dict:
    if tr.empty:
        return {"Trades": 0}
    net = tr["net_usd"].to_numpy()
    eq = np.concatenate([[0.0], np.cumsum(net)])
    monthly = tr.groupby(pd.to_datetime(tr["date"]).dt.to_period("M"))["net_usd"].sum()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    return {
        "Trades": len(tr), "Days traded %": 100 * tr["date"].nunique() / max(n_days, 1),
        "Win %": 100 * (net > 0).mean(), "Ø net R": tr["net_R"].mean(), "Ø net USD": net.mean(),
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Net total USD": net.sum(), "Max drawdown USD": (np.maximum.accumulate(eq) - eq).max(),
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Long %": 100 * (tr["direction"] == 1).mean(),
    }


PERIODS = ["optimization", "validation", "test"]
days_per_period = {p: sum(d["period"] == p for d in days) for p in PERIODS}


def period_metrics(tr):
    return {p: metrics(tr[tr["period"] == p] if not tr.empty else tr, days_per_period[p]) for p in PERIODS}

## 5. The Article's Rules (no optimization)

Opening range 15 minutes, entry on the next open after a 5-minute close beyond the range, volume filter, stop 10 points
beyond the level, target 3R, false-breakout exit, both windows, long and short.

In [ ]:
t0 = time.time()
article_trades = backtest(ARTICLE_RULES, days)
article_table = pd.DataFrame(period_metrics(article_trades)).T
print(f"Simulated in {time.time() - t0:.1f} s")
display(article_table.round(3))
if not article_trades.empty:
    print("Exit reasons (all periods):", article_trades["exit_reason"].value_counts().to_dict())

## 6. Optimization over the Parameter Grid

Every combination is simulated once over all days; the key figures are computed separately for optimization, validation
and test. This takes a few minutes.

In [ ]:
keys = list(GRID)
combos = [dict(zip(keys, vals)) for vals in itertools.product(*GRID.values())]
rows = []
t0 = time.time()
for i, cfg in enumerate(combos):
    pm = period_metrics(backtest(cfg, days))
    row = dict(cfg)
    for p in PERIODS:
        for k in ["Trades", "Win %", "Ø net R", "Ø net USD", "Profit factor", "t-value", "Net total USD", "Max drawdown USD"]:
            row[f"{p[:3]} {k}"] = pm[p].get(k, np.nan)
    rows.append(row)
    if (i + 1) % 100 == 0 or i + 1 == len(combos):
        el = time.time() - t0
        print(f"{i + 1:>5}/{len(combos)} combinations, {el / 60:.1f} min, ETA {el / (i + 1) * (len(combos) - i - 1) / 60:.1f} min")
grid_df = pd.DataFrame(rows)
grid_df.to_csv(OUTPUT_DIR / "orb_grid_results.csv", sep=";", index=False)
print(f"✓ Grid results saved: {(OUTPUT_DIR / 'orb_grid_results.csv').resolve()}")

## 7. Selection (optimization + validation only)

Eligible: at least `MIN_TRADES_OPT` trades in the optimization period and `MIN_TRADES_VAL` in validation.
Score = **min(t-value optimization, t-value validation)** → a combination must work in **both** periods.
The table also shows the test columns, but the choice does not use them.

In [ ]:
eligible = grid_df[(grid_df["opt Trades"] >= MIN_TRADES_OPT) & (grid_df["val Trades"] >= MIN_TRADES_VAL)].copy()
eligible["score"] = eligible[["opt t-value", "val t-value"]].min(axis=1)
show_cols = keys + ["opt Trades", "opt Ø net R", "opt t-value", "val Trades", "val Ø net R", "val t-value",
                    "tes Trades", "tes Ø net R", "tes t-value", "tes Net total USD"]

print(f"{len(eligible):,} of {len(grid_df):,} combinations eligible")
print("\nTop 10 by optimization t-value only (classic over-fitting check):")
top_opt = eligible.sort_values("opt t-value", ascending=False).head(10)
display(top_opt[show_cols].round(3))
print(f"  -> of these 10, validation positive: {(top_opt['val Ø net R'] > 0).sum()}, test positive: {(top_opt['tes Ø net R'] > 0).sum()}")

print("\nTop 10 by score = min(t optimization, t validation):")
top_score = eligible.sort_values("score", ascending=False).head(10)
display(top_score[show_cols + ["score"]].round(3))

best_cfg = {k: top_score.iloc[0][k] for k in keys}
best_cfg["or_minutes"] = int(best_cfg["or_minutes"])
best_cfg["volume_filter"] = bool(best_cfg["volume_filter"])
best_cfg["false_breakout_exit"] = bool(best_cfg["false_breakout_exit"])
print("\nChosen combination:", best_cfg)

## 8. Test Period (applied once)

The chosen combination vs. the article's rules vs. random direction on exactly the same entries (same stop distance and
target rule, no false-breakout exit for the random version). Also: how many of **all** combinations would have been
profitable in the test – the share expected by chance alone is about half.

In [ ]:
test_days = [d for d in days if d["period"] == "test"]
n_test_days = len(test_days)
best_trades = backtest(best_cfg, days)
rows = [
    {"Strategy": "Optimized (chosen on opt + val)", **metrics(best_trades[best_trades["period"] == "test"], n_test_days)},
    {"Strategy": "Article rules", **metrics(article_trades[article_trades["period"] == "test"], n_test_days)},
]
rand = [metrics(backtest(best_cfg, test_days, "random", seed), n_test_days) for seed in range(RANDOM_RUNS)]
rand_row = {"Strategy": f"Random direction, same entries (Ø {RANDOM_RUNS})"}
for k in rand[0]:
    vals = [r.get(k) for r in rand]
    rand_row[k] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
rows.append(rand_row)
test_table = pd.DataFrame(rows)
print(f"Test period: {n_test_days} trading days | costs {COST_PER_TRADE_POINTS} points per round trip, "
      f"{POINT_VALUE_USD} USD per point")
display(test_table.round(3))

print("\nCost sensitivity of the chosen combination in the test period (per round trip):")
bt_test = best_trades[best_trades["period"] == "test"]
sens = []
for cost in COST_SENSITIVITY_POINTS:
    adj = bt_test.copy()
    adj["net_points"] = adj["points"] - cost
    adj["net_R"] = adj["net_points"] / adj["risk_points"]
    adj["net_usd"] = adj["net_points"] * POINT_VALUE_USD
    sens.append({"Costs (points)": cost, **metrics(adj, n_test_days)})
display(pd.DataFrame(sens).round(3))

print("\nChosen combination in all periods:")
display(pd.DataFrame(period_metrics(best_trades)).T.round(3))

pos_share = (grid_df["tes Ø net R"] > 0).mean() * 100
sig_share = (grid_df["tes t-value"] >= 2).mean() * 100
print(f"\nAll {len(grid_df):,} combinations in the test period: {pos_share:.1f} % profitable, "
      f"{sig_share:.1f} % with t-value ≥ 2 (≈ 2–3 % would be expected by chance)")

### 8.1 Monthly Results and Equity Curve of the Chosen Combination

In [ ]:
bt = best_trades.copy()
bt["month"] = pd.to_datetime(bt["date"]).dt.to_period("M")
monthly = bt.groupby(["period", "month"])["net_usd"].agg(["count", "sum"]).rename(columns={"count": "trades", "sum": "net USD"})
display(monthly[monthly.index.get_level_values(0) == "test"].round(0))

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(14, 4))
    bt["equity"] = bt["net_usd"].cumsum()
    for p, col in zip(PERIODS, ["tab:blue", "tab:orange", "tab:green"]):
        part = bt[bt["period"] == p]
        ax.plot(pd.to_datetime(part["date"]), part["equity"], label=p, color=col)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title("Cumulative net USD (1 contract) – chosen combination")
    ax.legend()
    plt.show()
except ImportError:
    print("(matplotlib not installed – chart skipped)")

bt.drop(columns=["month", "equity"], errors="ignore").to_csv(OUTPUT_DIR / "orb_trades_chosen.csv", sep=";", index=False)
article_trades.to_csv(OUTPUT_DIR / "orb_trades_article_rules.csv", sep=";", index=False)
print(f"✓ Trades saved in {OUTPUT_DIR.resolve()}")

### 8.2 Which Parameters Matter? (average test result per parameter value)

If a parameter value is better on average across all combinations in **optimization and test**, that is a robust effect;
if it only helps in one period, it is noise.

In [ ]:
for k in keys:
    t = grid_df.groupby(k)[["opt Ø net R", "val Ø net R", "tes Ø net R"]].mean()
    print(f"\n{k}:")
    display(t.round(4))